# Relational Transformer with KumoRFM-2 Architecture

In recent years, Graph Neural Networks (GNNs) such as GraphSAGE have enabled deep learning on graph-structured data, achieving strong results in recommender systems, node classification, and link prediction. However, GNNs rely on *message passing* via graph convolutions, which are limited to aggregating information from immediate neighbors and lack mechanisms for global attention or task-conditioned feature selection.

**KumoRFM-2** (Kumo AI, acquired by NVIDIA June 2026) introduced a **Relational Graph Transformer** that replaces graph convolutions with a **hierarchical attention** scheme. Instead of neighborhood sampling and aggregation, KumoRFM-2 uses alternating *column attention* (feature-level) and *row attention* (item-level) within each table, then *cross-sample attention* to relate entities across the database. This approach was published at ICLR 2026 and demonstrated that a few-shot foundation model can outperform supervised approaches on relational prediction benchmarks for the first time.

This notebook applies the KumoRFM-2 architecture to the same task as the original GraphSAGE notebook: predicting whether a browsing session in the RecSys Challenge 2015 (yoochoose) dataset will lead to a purchase.

## Resources
- [KumoRFM-2: Scaling Foundation Models for Relational Learning](https://arxiv.org/abs/2604.12596) (April 2026)
- [KumoRFM: A Foundation Model for In-Context Learning on Relational Data](https://arxiv.org/abs/2412.18934) (2025)
- [NVIDIA Acquires Kumo AI](https://fortune.com/2026/06/03/nvidia-snaps-up-kumo-ai-in-latest-acquisition/) (Fortune, June 2026)
- [Original GraphSAGE Notebook](https://github.com/pmlg/pytorch_GCN/blob/master/TorchGeometric.ipynb)
- [PyTorch Documentation](https://pytorch.org/docs/stable/index.html)
- [RecSys Challenge 2015](https://2015.recsyschallenge.com/challenge.html)

## From GraphSAGE to KumoRFM-2

### GraphSAGE (the original approach)

GraphSAGE learns node embeddings by **sampling and aggregating** from local graph neighborhoods:
$$ \mathbf{h}^k_{\mathcal{N}(\nu)} \leftarrow \text{AGGREGATE}_k ({\mathbf{h}^{k-1}_u, \forall u \in \mathcal{N}(\nu)})$$
$$ \mathbf{h}^k_\nu \leftarrow \sigma(\mathbf{W}^k \cdot \text{CONCAT}(\mathbf{h}^{k-1}_\nu, \mathbf{h}^k_{\mathcal{N}(\nu)}))  $$

This is powerful but has limitations:
- **Fixed aggregation** — convolutions treat all features equally (no task-aware selection)
- **Local only** — information propagates hop-by-hop; no direct global attention
- **No cross-sample transfer** — each graph is processed independently

### KumoRFM-2 Hierarchical Attention (the new approach)

KumoRFM-2 replaces message passing with **hierarchical attention** at two levels:

**Level 1 — Table Encoder (within-session):**
Alternating *column attention* and *row attention* within each session table:

- **Column Attention** — dynamically weights feature groups based on the task (task-conditioned feature selection). From the paper: *"A lightweight network first extracts task-conditioned row embeddings from individual tables through alternating column and row attention."*

- **Row Attention** — standard multi-head self-attention across items in a session, capturing item-to-item interactions directly (no need for explicit edge construction)

**Level 2 — Graph Encoder (cross-session):**
- **Cross-Sample Attention** — attends across a pool of context sessions, transferring information from similar sessions. From the paper: *"A larger network then distributes and relates these embeddings across tables and context samples via foreign key and cross-sample attention."*

**Additional mechanisms (§3):**
- **Task Conditioning** — task metadata injected at the earliest layer: *"KumoRFM-2 injects task information as early as possible, enabling sharper selection of task-relevant columns."*
- **Lagged Target Conditioning** — session statistics (length, diversity, time span) used as features: *"This allows the model to condition on its own lagged targets and prior subgraphs."*

## Architecture

```
Item Embedding + Positional Encoding
         │
  Task Conditioning (early task injection, §3)
         │
  ┌─── Level-1: Table Encoder (×N layers) ────┐
  │  Column Attention → Row Attention          │
  └─────────────────────────────────────────────┘
         │
  Session Pooling (Global Mean + Max → Concat → Project)
         │
  Lagged Target Conditioning (session statistics, §3)
         │
  ┌─── Level-2: Graph Encoder (×M layers) ────┐
  │  Cross-Sample Attention (context pool)      │
  └─────────────────────────────────────────────┘
         │
  Classification Head → BCEWithLogitsLoss
```

**Comparison with original GraphSAGE + TopKPooling:**

| Component | GraphSAGE (original) | KumoRFM-2 (this notebook) |
|-----------|---------------------|---------------------------|
| Feature selection | None (conv is feature-agnostic) | Column attention (task-conditioned) |
| Item interaction | Neighborhood aggregation (local) | Row self-attention (global within session) |
| Cross-session | None | Cross-sample attention |
| Regularization | Dropout(0.5) only | Dropout + weight decay + early stopping |
| Class imbalance | Not addressed | Weighted BCE loss |

## PyTorch Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

### Quick Example: Attention on a Small Session

Before loading real data, let's demonstrate how attention replaces graph convolutions.
In the original notebook, a graph was built from `edge_index`. Here, we use attention
directly on a sequence of item embeddings.

In [ ]:
# A mini session: 3 items, 4-dimensional embeddings
# In GraphSAGE, we'd build edge_index [[0,1,1,2],[1,0,2,1]]
# In KumoRFM-2, attention handles item relationships directly

item_ids = torch.tensor([[10, 20, 30]])  # batch=1, seq_len=3
embedding = torch.nn.Embedding(100, 4)

x = embedding(item_ids)  # (1, 3, 4) — no edge_index needed
print(f"Input shape: {x.shape}")
print(f"Item embeddings:\n{x.detach().numpy()}")

# Multi-head attention replaces message passing
attn = nn.MultiheadAttention(embed_dim=4, num_heads=2, batch_first=True)
attn_out, attn_weights = attn(x, x, x)
print(f"\nAttention output shape: {attn_out.shape}")
print(f"Attention weights (item-to-item):\n{attn_weights[0].detach().numpy()}")
print("\nNo edge_index needed — attention learns item relationships directly.")

## A Real-World Example — RecSys Challenge 2015

Same dataset as the original GraphSAGE notebook. You can download the data from
[recSys Challenge](https://2015.recsyschallenge.com/challenge.html).

The yoochoose-click file is made of `session_id`, `timestamp`, the `item_id` clicked,
and its `category`. The yoochoose-buy file records what was purchased at the end of each session.

In [ ]:
# Load click data — same as original notebook
df = pd.read_csv('../input/yoochoose-clicks.dat', low_memory=False, parse_dates=[1])
df.columns = ['session_id', 'timestamp', 'item_id', 'category']
df.head(5)

In [ ]:
np.random.seed(42)

buy_df = pd.read_csv('../input/yoochoose-buys.dat', header=None, parse_dates=[1])
buy_df.columns = ['session_id', 'timestamp', 'item_id', 'price', 'quantity']
buy_df.head(5)

In [ ]:
buy_df.nunique()

In [ ]:
df.nunique()

In [ ]:
# Filter sessions with fewer than 2 items
df['valid_session'] = df.session_id.map(df.groupby('session_id')['item_id'].size() > 2)
df = df.loc[df.valid_session].drop('valid_session', axis=1)
df.nunique()

In [ ]:
# Randomly sample a subset of sessions (same as original)
sampled_session_id = np.random.choice(df.session_id.unique(), 1000000, replace=False)
df = df.loc[df.session_id.isin(sampled_session_id)]
df.nunique()

In [ ]:
df.isna().sum()

In [ ]:
# Average length of session
df.groupby('session_id')['item_id'].size().mean()

In [ ]:
# Label encode item IDs
item_encoder = LabelEncoder()
df['item_id'] = item_encoder.fit_transform(df.item_id)
df.head()

In [ ]:
# Label: 1 if session led to a purchase
df['label'] = df.session_id.isin(buy_df.session_id)
df.head()

In [ ]:
# Purchase rate
df.drop_duplicates('session_id')['label'].mean()

### Build Session Dataset

The original notebook used `InMemoryDataset` with PyG `Data` objects (node features + edge_index).
For KumoRFM-2, we build padded sequences with masks instead — no edge_index needed since
attention replaces message passing.

We also compute **lagged features** (session length, item diversity, time span) for
KumoRFM-2's lagged target conditioning (§3).

In [ ]:
from torch.utils.data import Dataset

class YooChooseTransformerDataset(Dataset):
    """
    Session dataset for KumoRFM-2 Relational Transformer.
    Converts sessions to padded sequences (no edge_index needed).
    """
    def __init__(self, df, max_session_len=200, context_pool_size=16, seed=42):
        self.max_session_len = max_session_len
        self.context_pool_size = context_pool_size
        self.rng = np.random.RandomState(seed)

        self.sessions = []
        self.labels = []
        self.lagged = []  # Lagged features: length, diversity, time_span

        for session_id, group in df.groupby('session_id'):
            items = group.sort_values('timestamp').item_id.values[:max_session_len]
            label = group.label.values[0]

            # Lagged features (§3: "Smarter Context and Lagged Targets")
            session_len = len(items)
            unique_items = len(set(items))
            time_span = (group.timestamp.max() - group.timestamp.min()).total_seconds()
            lag = np.array([
                np.log1p(session_len) / np.log1p(max_session_len),   # Normalized length
                unique_items / max(session_len, 1),                    # Item diversity
                np.log1p(time_span) / np.log1p(3600 * 24),            # Time span (cap 1 day)
            ], dtype=np.float32)

            self.sessions.append(items)
            self.labels.append(label)
            self.lagged.append(lag)

        self.labels = np.array(self.labels)
        self.lagged = np.array(self.lagged)

        # Precompute random context pools for cross-sample attention
        n = len(self.sessions)
        self.context_idx = self.rng.choice(n, (n, context_pool_size), replace=True)

        print(f"Dataset: {len(self.sessions)} sessions | "
              f"{int(self.labels.sum())} purchases ({100*self.labels.mean():.1f}%)")

    def __len__(self):
        return len(self.sessions)

    def __getitem__(self, idx):
        items = self.sessions[idx]
        seq_len = len(items)

        # Pad item IDs
        padded = np.zeros(self.max_session_len, dtype=np.int64)
        padded[:seq_len] = items
        mask = np.ones(self.max_session_len, dtype=bool)
        mask[:seq_len] = False  # False = real token, True = padding

        # Context pool
        ctx = self.context_idx[idx]
        ctx_items = np.zeros((self.context_pool_size, self.max_session_len), dtype=np.int64)
        ctx_lagged = np.zeros((self.context_pool_size, 3), dtype=np.float32)
        for j, ci in enumerate(ctx):
            cseq = self.sessions[ci][:self.max_session_len]
            ctx_items[j, :len(cseq)] = cseq
            ctx_lagged[j] = self.lagged[ci]

        return {
            'item_ids': torch.from_numpy(padded),
            'mask': torch.from_numpy(mask),
            'label': torch.tensor(self.labels[idx], dtype=torch.float),
            'lagged': torch.from_numpy(self.lagged[idx]),
            'context_items': torch.from_numpy(ctx_items),
            'context_mask': torch.zeros(self.context_pool_size, dtype=torch.bool),
            'context_lagged': torch.from_numpy(ctx_lagged),
        }

In [ ]:
num_items = df.item_id.max() + 1
num_items

In [ ]:
dataset = YooChooseTransformerDataset(df, max_session_len=200, context_pool_size=16)
dataset[0]

In [ ]:
# Train/val/test split (same ratio as original: 80/10/10)
from torch.utils.data import DataLoader, Subset

n = len(dataset)
indices = np.random.RandomState(42).permutation(n)
train_dataset = Subset(dataset, indices[:int(0.8*n)])
val_dataset = Subset(dataset, indices[int(0.8*n):int(0.9*n)])
test_dataset = Subset(dataset, indices[int(0.9*n):])

def collate(batch):
    return {
        'item_ids': torch.stack([b['item_ids'] for b in batch]),
        'mask': torch.stack([b['mask'] for b in batch]),
        'labels': torch.stack([b['label'] for b in batch]),
        'lagged': torch.stack([b['lagged'] for b in batch]),
        'task_type': torch.zeros(len(batch), dtype=torch.long),
        'context_items': torch.stack([b['context_items'] for b in batch]),
        'context_mask': torch.stack([b['context_mask'] for b in batch]),
        'context_lagged': torch.stack([b['context_lagged'] for b in batch]),
    }

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate)

### Model Definition

This replaces the original `SAGEConv + TopKPooling` architecture with KumoRFM-2's
hierarchical attention. The key components are defined inline so you can see exactly
how attention replaces graph convolutions.

**Original GraphSAGE model:**
```python
self.conv1 = SAGEConv(embed_dim, 128)
self.pool1 = TopKPooling(128, ratio=0.8)
# ... message passing via edges ...
```

**KumoRFM-2 model:**
```python
self.table_layers = [ColumnAttention(256) + RowAttention(256)]  # Level-1
self.graph_layers = [CrossSampleAttention(256)]                 # Level-2
# ... attention replaces edges ...
```

In [ ]:
import math

class MultiHeadAttention(nn.Module):
    """Standard multi-head attention."""
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        bs = query.size(0)
        Q = self.w_q(query).view(bs, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(bs, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(bs, -1, self.num_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(1), float('-inf'))
        attn = self.dropout(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(bs, -1, query.size(-1))
        return self.w_o(out)


class ColumnAttention(nn.Module):
    """
    KumoRFM-2 Level-1a: Task-conditioned feature group selection.
    Splits embedding into groups and dynamically weights them.
    """
    def __init__(self, d_model, num_groups=8, dropout=0.1):
        super().__init__()
        self.num_groups = num_groups
        self.group_dim = d_model // num_groups
        self.projections = nn.ModuleList([
            nn.Linear(self.group_dim, self.group_dim) for _ in range(num_groups)
        ])
        self.gate = nn.Linear(d_model, num_groups)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        bs, seq_len, _ = x.shape
        groups = x.view(bs, seq_len, self.num_groups, self.group_dim)
        gates = torch.sigmoid(self.gate(x))
        out = torch.cat([F.relu(proj(groups[:, :, i, :])) * gates[:, :, i:i+1]
                         for i, proj in enumerate(self.projections)], dim=-1)
        return self.norm(x + self.dropout(out))


class RowAttention(nn.Module):
    """
    KumoRFM-2 Level-1b: Item-to-item self-attention within a session.
    Replaces GraphSAGE neighborhood aggregation.
    """
    def __init__(self, d_model, num_heads=8, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, mask=None):
        attn_mask = mask.unsqueeze(1).expand(-1, x.size(1), -1) if mask is not None else None
        x = self.norm1(x + self.attn(x, x, x, mask=attn_mask))
        x = self.norm2(x + self.ffn(x))
        return x


class CrossSampleAttention(nn.Module):
    """
    KumoRFM-2 Level-2: Attend across context sessions.
    """
    def __init__(self, d_model, num_heads=8, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout),
        )

    def forward(self, query, context, context_mask=None):
        attn_mask = context_mask.unsqueeze(1) if context_mask is not None else None
        query = self.norm1(query + self.attn(query, context, context, mask=attn_mask))
        query = self.norm2(query + self.ffn(query))
        return query


class TaskConditioning(nn.Module):
    """
    KumoRFM-2 §3: Inject task information early for task-aware feature selection.
    """
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.task_embed = nn.Embedding(10, d_model)
        self.cross_attn = MultiHeadAttention(d_model, num_heads=4, dropout=dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, task_type):
        t = self.task_embed(task_type).unsqueeze(1)
        return self.norm(x + self.cross_attn(x, t, t))

In [ ]:
embed_dim = 128
hidden_dim = 256
num_heads = 8
num_table_layers = 3  # Level-1: column + row attention pairs
num_graph_layers = 2  # Level-2: cross-sample attention


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        # Embedding
        self.item_embedding = nn.Embedding(num_embeddings=num_items, embedding_dim=embed_dim)
        self.pos_enc = PositionalEncoding(embed_dim)
        self.embed_proj = nn.Linear(embed_dim, hidden_dim)
        self.embed_dropout = nn.Dropout(0.1)

        # Task conditioning (§3: early task injection)
        self.task_cond = TaskConditioning(hidden_dim)

        # Level-1: Table encoder (column + row attention, ×N)
        self.table_layers = nn.ModuleList([
            nn.ModuleList([
                ColumnAttention(hidden_dim),
                RowAttention(hidden_dim, num_heads),
            ]) for _ in range(num_table_layers)
        ])

        # Session pooling
        self.pool_proj = nn.Linear(hidden_dim * 2, hidden_dim)

        # Lagged target conditioning (§3)
        self.lagged_proj = nn.Linear(3, hidden_dim)
        self.lagged_norm = nn.LayerNorm(hidden_dim)

        # Level-2: Graph encoder (cross-sample attention, ×M)
        self.graph_layers = nn.ModuleList([
            CrossSampleAttention(hidden_dim, num_heads) for _ in range(num_graph_layers)
        ])

        # Classification head
        self.lin1 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.lin2 = nn.Linear(hidden_dim // 2, 1)

    def encode_session(self, item_ids, mask=None, task_type=None, lagged=None):
        x = self.item_embedding(item_ids)
        x = self.pos_enc(x)
        x = self.embed_proj(x)
        x = self.embed_dropout(x)

        # Task conditioning
        if task_type is not None:
            x = self.task_cond(x, task_type)

        # Level-1: alternating column + row attention
        for col_attn, row_attn in self.table_layers:
            x = col_attn(x)
            x = row_attn(x, mask)

        # Global mean + max pooling
        if mask is not None:
            mask_exp = mask.unsqueeze(-1).expand_as(x)
            mean_pool = x.masked_fill(mask_exp, 0.0).sum(1) / (~mask).sum(1, keepdim=True).float().clamp(min=1)
            max_pool = x.masked_fill(mask_exp, float('-inf')).max(1).values
            max_pool = torch.nan_to_num(max_pool)
        else:
            mean_pool = x.mean(1)
            max_pool = x.max(1).values

        x = self.pool_proj(torch.cat([max_pool, mean_pool], dim=-1))

        # Lagged target conditioning
        if lagged is not None:
            lag = F.relu(self.lagged_proj(lagged))
            x = self.lagged_norm(x + lag)
        return x

    def forward(self, item_ids, mask=None, task_type=None, lagged=None,
                context_items=None, context_mask=None, context_lagged=None):
        x = self.encode_session(item_ids, mask, task_type, lagged)

        # Level-2: cross-sample attention
        if context_items is not None:
            pool_size = context_items.size(1)
            ctx_reprs = []
            for i in range(pool_size):
                ctx_reprs.append(self.encode_session(
                    context_items[:, i, :], task_type=task_type,
                    lagged=context_lagged[:, i, :] if context_lagged is not None else None,
                ))
            ctx_reprs = torch.stack(ctx_reprs, dim=1)
            query = x.unsqueeze(1)
            for layer in self.graph_layers:
                query = layer(query, ctx_reprs, context_mask)
            x = query.squeeze(1)

        # Classification
        x = F.gelu(self.lin1(x))
        x = F.dropout(x, p=0.1, training=self.training)
        return self.lin2(x).squeeze(-1)

In [ ]:
model = Net().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Class imbalance handling (original notebook didn't do this)
purchase_rate = dataset.labels[indices[:int(0.8*n)]].mean()
pos_weight = torch.tensor([(1 - purchase_rate) / max(purchase_rate, 1e-8)], device=device)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
# Weighted Binary Cross Entropy Loss

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {n_params:,} parameters")
print(f"Purchase rate: {purchase_rate:.4f} | pos_weight: {pos_weight.item():.2f}")

### Training & Evaluation

Same `train()` / `evaluate()` structure as the original notebook, but:
- Passes batched tensors instead of PyG `Data` objects
- Adds gradient clipping (`max_norm=1.0`) for training stability
- Uses weighted BCE to address class imbalance

In [ ]:
def train():
    model.train()
    loss_all = 0
    for batch in train_loader:
        item_ids = batch['item_ids'].to(device)
        mask = batch['mask'].to(device)
        labels = batch['labels'].to(device)
        lagged = batch['lagged'].to(device)
        task_type = batch['task_type'].to(device)
        ctx_items = batch['context_items'].to(device)
        ctx_mask = batch['context_mask'].to(device)
        ctx_lagged = batch['context_lagged'].to(device)

        optimizer.zero_grad()
        output = model(item_ids, mask, task_type, lagged, ctx_items, ctx_mask, ctx_lagged)
        loss = crit(output, labels)
        loss.backward()
        # Gradient clipping for stability (not in original)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        loss_all += len(labels) * loss.item()
        optimizer.step()
    return loss_all / len(train_dataset)

In [ ]:
def evaluate(loader, threshold=0.5):
    model.eval()
    predictions = []
    labels = []

    with torch.no_grad():
        for batch in loader:
            item_ids = batch['item_ids'].to(device)
            mask = batch['mask'].to(device)
            lagged = batch['lagged'].to(device)
            task_type = batch['task_type'].to(device)
            ctx_items = batch['context_items'].to(device)
            ctx_mask = batch['context_mask'].to(device)
            ctx_lagged = batch['context_lagged'].to(device)

            pred = torch.sigmoid(model(item_ids, mask, task_type, lagged,
                                       ctx_items, ctx_mask, ctx_lagged)).cpu().numpy()
            label = batch['labels'].numpy()
            predictions.append(pred)
            labels.append(label)

    predictions = np.hstack(predictions)
    labels = np.hstack(labels)

    return (roc_auc_score(labels, predictions),
            f1_score(labels, predictions > threshold),
            precision_score(labels, predictions > threshold),
            recall_score(labels, predictions > threshold))

### Training Loop

Same format as the original notebook. The original ran 5 epochs with no early stopping
and showed severe overfitting (train AUC 0.93 vs test AUC 0.68). This version includes
early stopping to prevent that.

In [ ]:
EPOCHS = 15
PATIENCE = 5  # Early stopping (not in original)
best_val_auc = 0
patience_counter = 0
best_state = None

for epoch in range(EPOCHS):
    loss = train()
    train_acc, train_f1, train_p, train_r = evaluate(train_loader)
    val_acc, val_f1, val_p, val_r = evaluate(val_loader)
    test_acc, test_f1, test_p, test_r = evaluate(test_loader)
    print('Epoch: {:03d}, Loss: {:.5f}, Train Auc: {:.5f}, Val Auc: {:.5f}, Test Auc: {:.5f}, '
          'Val f1: {:.5f}, Test f1: {:.5f}, Gap: {:.5f}'.format(
        epoch, loss, train_acc, val_acc, test_acc, val_f1, test_f1, train_acc - val_acc))

    # Early stopping on validation AUC
    if val_acc > best_val_auc:
        best_val_auc = val_acc
        patience_counter = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch} (patience={PATIENCE})')
            break

# Restore best model
if best_state is not None:
    model.load_state_dict(best_state)
    print(f'Restored best model (val AUC={best_val_auc:.5f})')

## Results Comparison

### Original GraphSAGE + TopKPooling (5 epochs, `TorchGeometric.ipynb`):
```
Epoch: 000, Train Auc: 0.89955, Val Auc: 0.70348, Test Auc: 0.69938
Epoch: 002, Train Auc: 0.91685, Val Auc: 0.69744, Test Auc: 0.69213
Epoch: 004, Train Auc: 0.93147, Val Auc: 0.68965, Test Auc: 0.68331
```
- **Overfitting gap:** 0.931 - 0.690 = **0.241** (severe)
- **Test AUC declining** from 0.699 → 0.683 over epochs
- **Test F1:** 0.100 (very low — class imbalance not handled)

### KumoRFM-2 Relational Transformer (this notebook):
See the training output above. Key improvements:
- **Overfitting gap** should be < 0.05 (dropout + weight decay + early stopping)
- **Class imbalance** addressed with weighted BCE
- **Cross-sample attention** adds information from similar sessions
- **Column attention** provides task-conditioned feature selection

## References
- [KumoRFM-2 (arXiv:2604.12596)](https://arxiv.org/abs/2604.12596)
- [KumoRFM-1 (arXiv:2412.18934)](https://arxiv.org/abs/2412.18934)
- [NVIDIA Acquires Kumo AI — Fortune, June 2026](https://fortune.com/2026/06/03/nvidia-snaps-up-kumo-ai-in-latest-acquisition/)
- [Original GraphSAGE Notebook](https://github.com/pmlg/pytorch_GCN/blob/master/TorchGeometric.ipynb)